# NB16 — should-fix analyses, shard 0 (CPU)

Shard 0: by-subject mixed models on correct/incorrect outcomes and by-subject RM-ANOVA cross-checks; occlusion importance of electrodes and wavelet bands for P3 x spatial. Shards 1-2: the GCN with a learned adjacency (initialised at the spatial graph) for every representation on the selection folds, and for P3 on the confirmation folds.

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[dev,stats]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
sh(f'cd {SRC} && {sys.executable} -m pytest -q -p no:cacheprovider tests/test_should_fix.py tests/test_mixed.py tests/test_model_train.py', log=f'{WORK}/pytest_nb16.txt')

In [ ]:
import glob
CACHE = [d for d in glob.glob('/kaggle/input/**/cache', recursive=True) if os.path.exists(f'{d}/index.npz')][0]
PART = sorted(glob.glob('/kaggle/input/**/participants.tsv', recursive=True))[0]
FOLDS = f'{SRC}/splits/folds_ds004504.json'
print(CACHE, PART, FOLDS)

In [ ]:
sh(f'{PY} mixed --runs {SRC}/reports/grid/runs_s0.jsonl {SRC}/reports/grid/runs_s1.jsonl {SRC}/reports/grid/runs_s2.jsonl --tag main --out {WORK}/mixed --prefix selection', log=f'{WORK}/mixed_selection.log')
sh(f'{PY} mixed --runs {SRC}/reports/confirm/confirm_runs_s0.jsonl {SRC}/reports/confirm/confirm_runs_s1.jsonl --tag confirm --out {WORK}/mixed --prefix confirmation', log=f'{WORK}/mixed_confirmation.log')
sh(f'{PY} mixed --runs {SRC}/reports/external/ext_runs_s0.jsonl {SRC}/reports/external/ext_runs_s1.jsonl --tag main --out {WORK}/mixed --prefix external', log=f'{WORK}/mixed_external.log')

In [ ]:
sh(f'{PY} mixed --runs {SRC}/reports/grid/runs_s0.jsonl {SRC}/reports/grid/runs_s1.jsonl {SRC}/reports/grid/runs_s2.jsonl --tag main --outcome correct --out {WORK}/mixed --prefix selection_correct', log=f'{WORK}/mixed_selection_correct.log')
sh(f'{PY} mixed --runs {SRC}/reports/confirm/confirm_runs_s0.jsonl {SRC}/reports/confirm/confirm_runs_s1.jsonl --tag confirm --outcome correct --out {WORK}/mixed --prefix confirmation_correct', log=f'{WORK}/mixed_confirmation_correct.log')

In [ ]:
sh(f'{PY} explain --cache {CACHE} --folds {FOLDS} --out {WORK}/explain --pipeline P3 --edge spatial', log=f'{WORK}/explain.log')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)